In [2]:
import pandas as pd 
import numpy as np 

Nolasam datus no failiem:
- PIF_kontakti.csv
- The Table.csv
- pif_age_gender.csv
- Kavejuma_datums.xlsx

In [3]:
# Visas aktivitātes, ar V.U., p.k. un lietas numuriem
events_path = "G://GSCLV-FIN//Current month BI Reports//Client profile//2026//Valuation data//PIF_kontakti.csv"
events_df = pd.read_csv(events_path, sep=';')

# Automātiski atjaunojams fails ar datiem par visiem klientiem 
clients_path = "G://GSCLV-FIN PUBLIC//-=DATI=-//Report Automatization TheTable//The Table.csv"
clients_df = pd.read_csv(clients_path, sep=';')

# R ģenerēts fails ar klientu dzimumu un vecumu:
age_gender_path = "G:\\GSCLV-FIN\\Current month BI Reports\\Client profile\\2026\\Valuation data\\pif_age_gender.csv"
age_gender_df = pd.read_csv(age_gender_path, sep=';')

# kavējuma datumu fails 
delay_path = "G:\\GSCLV-FIN\\Current month BI Reports\\Client profile\\2026\\Valuation data\\Kavejuma_datums.xlsx"
delay_df = pd.read_excel(delay_path)

C:\Users\anastasija.v\AppData\Local\Temp\ipykernel_21932\1789040504.py:3: DtypeWarning: Columns (0: Attachment, 1: Home Phone, 2: Contact) have mixed types. Specify dtype option on import or set low_memory=False.
  events_df = pd.read_csv(events_path, sep=';')
C:\Users\anastasija.v\AppData\Local\Temp\ipykernel_21932\1789040504.py:7: DtypeWarning: Columns (0: Generation, 1: Street number, 2: Addr 1, 3: DOB, 4: Home, 5: POE#, 6: Cell, 7: Other, 8: Street type) have mixed types. Specify dtype option on import or set low_memory=False.
  clients_df = pd.read_csv(clients_path, sep=';')


Atstājam tikai nepieciešamās kolonnas un pārveidojam formātus:

events_clean: 'File', 'Done date', 'Type', 'Description'
clients_clean: 'File',     
                'Client number', # client ID
                'Legal start', # for legal_flag
                'Principal', # debt amount
                'Agree.Date', # agreement date
                'Termin.date', # agreement termination date
                'Listed', # purchase date
                'LastPaym.', # last payment date
                'City', #  city/region
                'User 1', # service
                'Home', # is_phone flag
                'Cell',
                'Email', # is_email flag
                'ZIP' # region


In [3]:
events_clean = events_df[['File', 'Done date', 'Type', 'Description']]
events_clean['Done date'] = pd.to_datetime(events_clean['Done date'], errors='coerce', format='%d.%m.%Y')

In [14]:
# Atstāt noteiktas kolonnas clients_df:
clients_clean = clients_df[['File', 
                            'Client number', # client ID
                            'Legal start', # for legal_flag
                            'Principal', # debt amount
                            'Agree.Date', # agreement date
                            'Termin.date', # agreement termination date
                            'Listed', # purchase date
                            'LastPaym.', # last payment date
                            'City', #  city/region
                            'User 1', # service
                            'Home', # is_phone flag
                            'Cell',
                            'Email', # is_email flag
                            'ZIP' # region
                            ]]

In [15]:
# Apvienojam datus pa 'File' kolonnu, lai iegūtu vienu kopīgu datu kopu:
combined_df = pd.merge(age_gender_df, clients_clean, on='File', how='left')
combined_df = pd.merge(combined_df, delay_df, on='File', how='left')

In [16]:
# Pārbaudām, vai nav File dublikātu:
duplicate_files = combined_df['File'].duplicated().sum()
print(f"Number of duplicate 'File' entries: {duplicate_files}")

Number of duplicate 'File' entries: 0


In [17]:
# Apvienojam ar events_clean, lai iegūtu pilnīgu datu kopu:
final_df = pd.merge(combined_df, events_clean, on='File', how='left')


In [18]:
# Pārveidot datumu kolonnas datetime formātā:
final_df['Agree.Date'] = pd.to_datetime(final_df['Agree.Date'], errors='coerce', format='%d/%m/%Y')
final_df['Termin.date'] = pd.to_datetime(final_df['Termin.date'], errors='coerce', format='%d/%m/%Y')
final_df['Listed'] = pd.to_datetime(final_df['Listed'], errors='coerce', format='%d/%m/%Y') 

Pievienojam kolonnas:
- Vai ir legal?
- Vai ir epasts?
- Vai ir telefona numurs?

In [19]:
#Pievienot kolonnu vai ir Legal start datums, lai varētu izveidot legal_flag: legal vai non-legal:  
final_df['legal_flag'] = np.where(final_df['Legal start'].notna(), 'legal', 'non-legal')

In [20]:
# Pievienot kolonnu, vai ir email vai telefons, lai varētu izveidot contact_flag: contactable vai non-contactable:
final_df['contact_flag'] = np.where((final_df['Home'].notna()) | (final_df['Cell'].notna()) | (final_df['Email'].notna()), 'contactable', 'non-contactable')

Sadalam kopējo datu kopu uz divām: legal un non-legal:

In [21]:
legal_df = final_df[final_df['legal_flag'] == 'legal']
non_legal_df = final_df[final_df['legal_flag'] == 'non-legal']
# noņemt liekas kolonnas, kas vairs nav nepieciešamas:
columns_to_drop = ['Legal start', 'legal_flag', 'Home', 'Cell', 'Email']
non_legal_df = non_legal_df.drop(columns=columns_to_drop)
legal_df = legal_df.drop(columns=columns_to_drop)

In [12]:
# Aktivitāšu skaits pa lietām (File): 
activity_counts = non_legal_df.groupby('File').size().reset_index(name='Activity Count')
non_legal_df = non_legal_df.merge(activity_counts, on='File', how='left')
